# 1 — Export your trained Gemma 2 2B LoRA model for Kaggle

**Run this notebook locally, in the same Python environment used for training. Do not import this exporter into Kaggle.**

This prepares the already-trained model for **LLM Classification Finetuning**, not Gemma 4 Developer Agent.
It copies the best saved **adapter + classification head**, its tokenizer, the frozen base checkpoint from your local
Hugging Face cache, and Linux-compatible Python wheels. **It does not train, merge, or quantize the model. No GPU is needed.**

After it finishes, upload `arena_lora_2b_assets.zip` through **Kaggle → Create → New Dataset**, keeping it private.
Then import `2_kaggle_arena_submission.ipynb` and attach that dataset plus the competition data.

The weights are not stored inside an `.ipynb` file. Your log files alone cannot reconstruct them.
Keep the original run directory and your Hugging Face model cache intact. Internet is used here only for package-wheel downloads.

In [1]:
from pathlib import Path

# Prefilled from your completed run. Change only if you moved the output directory.
RUN_DIR = Path('C:\\Users\\yana\\Desktop\\Library\\AI\\ai-competitions\\tmp\\outputs\\arena_lora_2b_20261011_064837')

# Optional: the EXACT cached base-model snapshot used in training.
# Normally leave None; the exporter resolves the already-cached checkpoint, without downloading newer weights.
BASE_SNAPSHOT_DIR = None

# Common Kaggle Python runtimes. Only matching Linux wheels are used by the inference notebook.
KAGGLE_PYTHON_VERSIONS = ("3.11", "3.12", "3.13")

## Locate the completed run and check the saved head
The training notebook saved `adapter/` after `load_best_model_at_end=True`.
Use that folder, not a new head and not a randomly selected checkpoint. The safetensors check below fails early
when either the LoRA parameters or the three-class classification head is missing.

In [2]:
import hashlib
import json
import shutil
import subprocess
import sys
import warnings
import zipfile
from datetime import datetime
from importlib import metadata

from huggingface_hub import try_to_load_from_cache
from packaging.markers import default_environment
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
from safetensors import safe_open
from transformers import AutoTokenizer

if not RUN_DIR.is_dir():
    # Also support running from the original project folder or its parent.
    run_name = "arena_lora_2b_20261011_064837"
    candidates = [Path("../tmp/outputs") / run_name, Path("./tmp/outputs") / run_name]
    existing = [p.resolve() for p in candidates if p.is_dir()]
    if len(existing) != 1:
        raise FileNotFoundError("Set RUN_DIR to the completed training run containing adapter/ and run_summary.json.")
    RUN_DIR = existing[0]

run_summary = json.loads((RUN_DIR / "run_summary.json").read_text(encoding="utf-8"))
run_config = run_summary["config"]
assert run_config["model_name"] == "google/gemma-2-2b-it"
assert run_config["label_names"] == ["win_a", "win_b", "tie"]
assert (run_config["max_length"], run_config["prompt_budget"]) == (1536, 256)

ADAPTER_DIR = RUN_DIR / "adapter"
adapter_config = json.loads((ADAPTER_DIR / "adapter_config.json").read_text(encoding="utf-8"))
assert adapter_config["task_type"] == "SEQ_CLS"
assert adapter_config["peft_type"] == "LORA"
assert "score" in (adapter_config.get("modules_to_save") or [])
assert (ADAPTER_DIR / "tokenizer.json").is_file(), "The trained tokenizer is missing."
with safe_open(str(ADAPTER_DIR / "adapter_model.safetensors"), framework="pt", device="cpu") as f:
    keys = list(f.keys())
    head_keys = [k for k in keys if "score" in k and k.endswith("weight")]
    assert len(head_keys) == 1, f"Expected one saved head, found {head_keys}"
    assert f.get_slice(head_keys[0]).get_shape() == [3, 2304]
    assert any("lora_A" in k for k in keys) and any("lora_B" in k for k in keys)

EXPORT_DIR = RUN_DIR / f"kaggle_export_{datetime.now():%Y%m%d_%H%M%S}"
BUNDLE_DIR = EXPORT_DIR / "arena_lora_2b_assets"
BUNDLE_DIR.mkdir(parents=True, exist_ok=False)
print("Source run:", RUN_DIR.resolve())
print("Best saved validation log loss:", run_summary["best_checkpoint_log_loss"])
print("Export directory:", EXPORT_DIR.resolve())

c:\Users\yana\Desktop\Library\AI\ai-competitions\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Source run: C:\Users\yana\Desktop\Library\AI\ai-competitions\tmp\outputs\arena_lora_2b_20261011_064837
Best saved validation log loss: 0.9646593046935478
Export directory: C:\Users\yana\Desktop\Library\AI\ai-competitions\tmp\outputs\arena_lora_2b_20261011_064837\kaggle_export_20261011_193259


## Copy the cached base checkpoint and adapter
Your run recorded `model_revision=null`, so it did not record an immutable base-model commit.
The exporter therefore locates **the cached `config.json`** to identify the snapshot. It does not require unrelated cached files such as `README.md` or `generation_config.json`. It checks that all **necessary weight shards** are present before copying anything.
It deliberately does not fetch a possibly changed `main` checkpoint from the internet.
If you changed the cache after training, set `BASE_SNAPSHOT_DIR` to the training-time snapshot before exporting.
The original checkpoint files are copied, not rewritten.

**Fix for newer `huggingface_hub`:** `snapshot_download(local_files_only=True)` may raise `IncompleteSnapshotError` when optional repository files are not cached. This exporter avoids that whole-repository check.

In [3]:
if BASE_SNAPSHOT_DIR is None:
    # Resolve one cached file rather than requesting a COMPLETE Hub snapshot.
    # The cache can omit README, generation_config, etc. and still contain
    # every weight needed to load the sequence-classification backbone.
    revision = run_config.get("model_revision") or adapter_config.get("revision") or "main"
    cached_config = try_to_load_from_cache(
        repo_id=run_config["model_name"], filename="config.json", revision=revision,
    )
    if not isinstance(cached_config, str) or not Path(cached_config).is_file():
        raise FileNotFoundError(
            "The base model's config.json is not available in the local Hugging Face cache. "
            "Set BASE_SNAPSHOT_DIR to the snapshot used for training."
        )
    BASE_SNAPSHOT_DIR = Path(cached_config).parent.resolve()
else:
    BASE_SNAPSHOT_DIR = Path(BASE_SNAPSHOT_DIR).expanduser().resolve()

base_config_path = BASE_SNAPSHOT_DIR / "config.json"
base_config = json.loads(base_config_path.read_text(encoding="utf-8"))
assert base_config["model_type"] == "gemma2" and base_config["hidden_size"] == 2304
index_path = BASE_SNAPSHOT_DIR / "model.safetensors.index.json"
if index_path.is_file():
    weight_names = sorted(set(json.loads(index_path.read_text())["weight_map"].values()))
    base_names = ["config.json", "model.safetensors.index.json", *weight_names]
else:
    base_names = ["config.json", "model.safetensors"]
for name in base_names:
    if not (BASE_SNAPSHOT_DIR / name).is_file():
        raise FileNotFoundError(f"Incomplete local checkpoint: {BASE_SNAPSHOT_DIR / name}")

(BUNDLE_DIR / "base_model").mkdir()
(BUNDLE_DIR / "adapter").mkdir()
for name in base_names:
    shutil.copy2(BASE_SNAPSHOT_DIR / name, BUNDLE_DIR / "base_model" / name)
# Carry any locally cached model notices along with the weights.
for pattern in ["LICENSE*", "NOTICE*", "README.md", "USE_POLICY*"]:
    for path in BASE_SNAPSHOT_DIR.glob(pattern):
        if path.is_file():
            shutil.copy2(path, BUNDLE_DIR / "base_model" / path.name)

adapter_names = [
    "adapter_config.json", "adapter_model.safetensors", "tokenizer.json", "tokenizer.model",
    "tokenizer_config.json", "special_tokens_map.json", "added_tokens.json", "chat_template.jinja",
]
for name in adapter_names:
    if (ADAPTER_DIR / name).is_file():
        shutil.copy2(ADAPTER_DIR / name, BUNDLE_DIR / "adapter" / name)

reference_file = RUN_DIR / "test_predictions.csv"
if reference_file.is_file():
    shutil.copy2(reference_file, BUNDLE_DIR / "public_test_reference.csv")

manifest = {
    "bundle_format": "arena-gemma2-lora-v1",
    "competition": "llm-classification-finetuning",
    "model_name": run_config["model_name"],
    "base_snapshot": BASE_SNAPSHOT_DIR.name,
    "revision_was_recorded_during_training": bool(run_config.get("model_revision")),
    "base_subdir": "base_model", "adapter_subdir": "adapter",
    "max_length": 1536, "prompt_budget": 256,
    "label_names": ["win_a", "win_b", "tie"],
    "label_columns": ["winner_model_a", "winner_model_b", "winner_tie"],
    "attention_backend": "eager", "quantized": False, "merged": False,
    "best_checkpoint_log_loss": run_summary["best_checkpoint_log_loss"],
    "reloaded_validation_log_loss": run_summary["validation"]["valid_log_loss"],
    "training_versions": run_config["versions"],
    "base_files": {name: (BUNDLE_DIR / "base_model" / name).stat().st_size for name in base_names},
}
print("Cached base snapshot:", BASE_SNAPSHOT_DIR)
print("Copied base and trained adapter; the original files are unchanged.")

Cached base snapshot: C:\Users\yana\.cache\huggingface\hub\models--google--gemma-2-2b-it\snapshots\299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8
Copied base and trained adapter; the original files are unchanged.


## Record tokenization checks
The following functions are copied from your executed training notebook, including JSON parsing, turn padding,
section text, BOS/EOS, the 256-token prompt cap, and answer head/tail truncation.
The Kaggle notebook checks fingerprints of synthetic examples against this saved tokenizer.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(str(BUNDLE_DIR / "adapter"), local_files_only=True)
tokenizer.padding_side = "right"
MAX_LENGTH = 1536
PROMPT_BUDGET = 256

In [5]:
def parse_turns(value):
    """Decode one competition field. Never evaluate Python code from the data."""
    turns = json.loads(value) if isinstance(value, str) else value
    if turns is None:
        return [""]
    if not isinstance(turns, list):
        raise ValueError("Expected a JSON list of turns.")
    if not all(turn is None or isinstance(turn, str) for turn in turns):
        raise ValueError("Turns must be strings or null.")
    return ["" if turn is None else turn for turn in turns] or [""]


def format_turns(turns):
    return "\n\n".join(
        f"[Turn {i + 1}]\n{turn if turn else '[No text]'}"
        for i, turn in enumerate(turns)
    )


def prepare_row(row):
    try:
        p, a, b = [parse_turns(row[col]) for col in ["prompt", "response_a", "response_b"]]
    except (ValueError, TypeError) as exc:
        raise ValueError(f"Invalid conversation in row id={row['id']}: {exc}") from exc
    # Group identical normalized full prompt conversations, not individual response pairs.
    normalized = json.dumps([" ".join(turn.split()) for turn in p], ensure_ascii=False)
    group = hashlib.sha256(normalized.encode("utf-8")).hexdigest()
    n_turns = max(len(p), len(a), len(b))
    mismatched = len({len(p), len(a), len(b)}) > 1
    p, a, b = [turns + [""] * (n_turns - len(turns)) for turns in [p, a, b]]
    return {
        "prompt_text": format_turns(p),
        "response_a_text": format_turns(a),
        "response_b_text": format_turns(b),
        "prompt_group": group,
        "num_turns": n_turns,
        "mismatched_turns": mismatched,
    }


def encode_text(text):
    return tokenizer.encode(text, add_special_tokens=False)


prefix_ids = encode_text("Choose which response a human would prefer: A, B, or tie.\n\n[Prompt]\n")
a_header_ids = encode_text("\n\n[Response A]\n")
b_header_ids = encode_text("\n\n[Response B]\n")
suffix_ids = encode_text("\n\n[Decision: A, B, or tie]")
omission_ids = encode_text("\n[... omitted ...]\n")
overhead = len(prefix_ids) + len(a_header_ids) + len(b_header_ids) + len(suffix_ids) + 2
assert MAX_LENGTH % 8 == 0 and MAX_LENGTH <= tokenizer.model_max_length
assert MAX_LENGTH - overhead - PROMPT_BUDGET > 2 * len(omission_ids)


def trim_tokens(ids, budget):
    """A token budget for ONE example, with head-and-tail preservation."""
    if len(ids) <= budget:
        return ids
    if budget <= len(omission_ids):
        return ids[:budget]
    remaining = budget - len(omission_ids)
    head = (remaining + 1) // 2
    tail = remaining // 2
    return ids[:head] + omission_ids + (ids[-tail:] if tail else [])


def assemble_input(p, a, b):
    p_budget = min(len(p), PROMPT_BUDGET)
    answer_space = MAX_LENGTH - overhead - p_budget
    a_budget = min(len(a), answer_space // 2)
    b_budget = min(len(b), answer_space // 2)
    # Redistribute unused answer space without ever dropping the other answer's allocation.
    spare = answer_space - a_budget - b_budget
    extra_a = min(spare, len(a) - a_budget)
    a_budget += extra_a
    b_budget += min(spare - extra_a, len(b) - b_budget)
    input_ids = (
        [tokenizer.bos_token_id] + prefix_ids + trim_tokens(p, p_budget)
        + a_header_ids + trim_tokens(a, a_budget)
        + b_header_ids + trim_tokens(b, b_budget)
        + suffix_ids + [tokenizer.eos_token_id]
    )
    assert len(input_ids) <= MAX_LENGTH
    return input_ids, [len(p) > p_budget, len(a) > a_budget, len(b) > b_budget]


def tokenize(batch):
    # Each of these is a batch of complete example strings; never slice the row list as a token limit.
    p_batch = tokenizer(batch["prompt_text"], add_special_tokens=False, truncation=False)["input_ids"]
    a_batch = tokenizer(batch["response_a_text"], add_special_tokens=False, truncation=False)["input_ids"]
    b_batch = tokenizer(batch["response_b_text"], add_special_tokens=False, truncation=False)["input_ids"]
    result = {key: [] for key in ["input_ids", "attention_mask", "length", "original_length",
                                  "truncated_prompt", "truncated_a", "truncated_b"]}
    for p, a, b in zip(p_batch, a_batch, b_batch):
        ids, flags = assemble_input(p, a, b)
        result["input_ids"].append(ids)
        result["attention_mask"].append([1] * len(ids))
        result["length"].append(len(ids))
        result["original_length"].append(overhead + len(p) + len(a) + len(b))
        for name, flag in zip(["truncated_prompt", "truncated_a", "truncated_b"], flags):
            result[name].append(flag)
    if "labels" in batch:
        result["labels"] = batch["labels"]
    return result

In [6]:
def encode_records(records, batch_size=128):
    """Encode raw CSV records with the training notebook's exact functions."""
    encoded = []
    for start in range(0, len(records), batch_size):
        prepared = [prepare_row(row) for row in records[start:start + batch_size]]
        batch = {key: [row[key] for row in prepared] for key in
                 ["prompt_text", "response_a_text", "response_b_text"]}
        result = tokenize(batch)
        encoded.extend({"input_ids": ids, "attention_mask": mask}
                       for ids, mask in zip(result["input_ids"], result["attention_mask"]))
    return encoded

# Synthetic examples; no training or hidden-test data is packaged here.
fixtures = [
    {"id": "short", "prompt": json.dumps(["What is 2 + 2?"]),
     "response_a": json.dumps(["4."]), "response_b": json.dumps(["5."])},
    {"id": "turns", "prompt": json.dumps(["Explain a loop.", "Show code: π / 한글"]),
     "response_a": json.dumps(["Repeat a step.", "for i in range(2):\n    print(i)"]),
     "response_b": json.dumps([None])},
    {"id": "long", "prompt": json.dumps(["Context: " + "alpha beta " * 1600]),
     "response_a": json.dumps(["Answer A: " + "gamma delta " * 1800]),
     "response_b": json.dumps(["Answer B: " + "epsilon zeta " * 1800])},
    {"id": "asymmetric", "prompt": json.dumps(["Compare these."]),
     "response_a": json.dumps(["Short."]),
     "response_b": json.dumps(["Long answer. " * 2500])},
]

fixture_inputs = encode_records(fixtures)
for row, encoded in zip(fixtures, fixture_inputs):
    row["expected_length"] = len(encoded["input_ids"])
    row["expected_sha256"] = hashlib.sha256(
        json.dumps(encoded["input_ids"], separators=(",", ":")).encode()
    ).hexdigest()
(BUNDLE_DIR / "tokenization_checks.json").write_text(json.dumps(fixtures, ensure_ascii=False), encoding="utf-8")
print("Saved tokenizer fingerprints for", len(fixtures), "synthetic examples.")

Saved tokenizer fingerprints for 4 synthetic examples.


## Download offline dependencies for Linux, not Windows
Your local environment and Kaggle need compatible model/tokenizer libraries. This cell downloads wheels
for the exact installed Transformers/PEFT stack and its non-PyTorch dependencies.
It does **not** reinstall local packages, include your credentials, or replace Kaggle's CUDA PyTorch installation.
PyTorch, NumPy, and pandas are supplied by Kaggle; their required version constraints are checked there.

Downloads require internet here. If a specific Python version cannot use the installed stack, its error is printed.
The archive will contain only successfully prepared runtime variants. If Kaggle uses a different Python version,
add it to `KAGGLE_PYTHON_VERSIONS` and export again. No source builds or Windows wheels are used.

In [7]:
SEED_PACKAGES = ["transformers", "peft", "accelerate", "huggingface_hub", "tokenizers", "safetensors"]
PRESERVE_KAGGLE = {"torch", "numpy", "pandas"}


def locked_requirements(python_version):
    environment = default_environment()
    environment.update({
        "os_name": "posix", "sys_platform": "linux", "platform_system": "Linux",
        "platform_machine": "x86_64", "python_version": python_version,
        "python_full_version": python_version + ".0", "implementation_version": python_version + ".0", "extra": "",
    })
    pending = list(SEED_PACKAGES)
    versions, preserved_requirements = {}, set()
    while pending:
        name = canonicalize_name(pending.pop())
        if name in versions or name in PRESERVE_KAGGLE:
            continue
        if name == "triton" or name.startswith("nvidia-"):
            raise RuntimeError(f"Unexpected CUDA dependency {name}; do not package a second CUDA stack.")
        installed = metadata.version(name)
        versions[name] = installed
        for requirement in metadata.requires(name) or []:
            req = Requirement(requirement)
            if req.marker is not None and not req.marker.evaluate(environment):
                continue
            dep = canonicalize_name(req.name)
            if dep in PRESERVE_KAGGLE:
                preserved_requirements.add(req.name + str(req.specifier))
            else:
                pending.append(dep)
    return [f"{name}=={v}" for name, v in sorted(versions.items())], sorted(preserved_requirements)


manifest["runtimes"] = {}
for py in KAGGLE_PYTHON_VERSIONS:
    wheel_dir = BUNDLE_DIR / "wheels" / py
    wheel_dir.mkdir(parents=True)
    requirements, preserved = locked_requirements(py)
    req_file = wheel_dir / "requirements.txt"
    req_file.write_text("\n".join(requirements) + "\n", encoding="utf-8")
    cmd = [sys.executable, "-m", "pip", "download", "--disable-pip-version-check", "--no-deps",
           "--only-binary=:all:", "--platform", "manylinux2014_x86_64",
           "--platform", "manylinux_2_28_x86_64", "--implementation", "cp",
           "--python-version", py, "--dest", str(wheel_dir), "-r", str(req_file)]
    print(f"Downloading Python {py} Linux wheels...")
    completed = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if completed.returncode:
        print(completed.stdout[-6000:])
        shutil.rmtree(wheel_dir)
        warnings.warn(f"Could not prepare Python {py}; this variant will not be included.")
        continue
    manifest["runtimes"][py] = {
        "wheel_subdir": f"wheels/{py}",
        "preserved_requirements": preserved,
    }
    print("Prepared", len(list(wheel_dir.glob("*.whl"))), "wheels.")

if not manifest["runtimes"]:
    raise RuntimeError("No Linux runtime could be prepared. Resolve the pip error above before uploading anything.")
print("Prepared Python variants:", list(manifest["runtimes"]))

Prepared 29 wheels.
Prepared 29 wheels.
Prepared 29 wheels.
Prepared Python variants: ['3.11', '3.12', '3.13']


## Create the private-dataset archive
Only model assets, package wheels, tokenizer checks, and public-example prediction references are included.
The export does not upload anything and does not include train.csv, test.csv, optimizer state, or authentication tokens.
The archive is several gigabytes because it includes the frozen base model as well as the small adapter.

In [8]:
manifest["adapter_sha256"] = hashlib.sha256((BUNDLE_DIR / "adapter" / "adapter_model.safetensors").read_bytes()).hexdigest()
(BUNDLE_DIR / "arena_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
(BUNDLE_DIR / "MODEL_NOTICE.txt").write_text(
    "Private inference bundle of google/gemma-2-2b-it with your trained LoRA adapter.\n"
    "Base model is subject to the Gemma terms: https://ai.google.dev/gemma/terms\n"
    "Do not publish or redistribute the base weights without complying with those terms.\n",
    encoding="utf-8",
)
archive = EXPORT_DIR / "arena_lora_2b_assets.zip"
# ZIP_STORED avoids spending time compressing large safetensors files.
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as z:
    for file in sorted(BUNDLE_DIR.rglob("*")):
        if file.is_file():
            z.write(file, file.relative_to(EXPORT_DIR).as_posix())
print("UPLOAD THIS ARCHIVE AS A PRIVATE KAGGLE DATASET:")
print(archive.resolve())
print(f"Archive size: {archive.stat().st_size / 2**30:.2f} GiB")
print("Next: import 2_kaggle_arena_submission.ipynb and attach this dataset and the competition data.")

UPLOAD THIS ARCHIVE AS A PRIVATE KAGGLE DATASET:
C:\Users\yana\Desktop\Library\AI\ai-competitions\tmp\outputs\arena_lora_2b_20261011_064837\kaggle_export_20261011_193259\arena_lora_2b_assets.zip
Archive size: 5.06 GiB
Next: import 2_kaggle_arena_submission.ipynb and attach this dataset and the competition data.


## References
[Competition requirements](https://www.kaggle.com/competitions/llm-classification-finetuning/overview)
· [PEFT checkpoint format](https://huggingface.co/docs/peft/developer_guides/checkpoint)
· [Offline Transformers loading](https://huggingface.co/docs/transformers/installation#offline-mode)
· [Cross-platform pip downloads](https://pip.pypa.io/en/stable/cli/pip_download/)

Prepared from your executed `arena_gemma2_2b_lora(1).ipynb` and `run_summary.json`.
The source run was completed for one epoch. Nothing in this exporter selects or trains a new model.